# Mission 2A: Wake Up Mary's Neurons

Epoch Zero NLP Bootcamp · Section 2: PyTorch and Text Classification with Neural

Networks · Basic track assignment

#### TRANSMISSION RECEIVED: SHIP HAIL MARY, DAY 214

Crew, good news and bad news. Thanks to your work in Section 1, Mary can finally
split sentences into words. The bad news: she still has no idea what those words
mean. Last night the hull alarm sounded, and Mary filed it under "dinner menu".
"Mary... hear words. Mary not know... danger? Or snack? Please. Give Mary... neurons."
Your mission: build Mary's very first neural network and teach her to tell DISTRESS
transmissions from ROUTINE ones, before the next alarm.

#### Mission briefing

Estimated time: About 2 hours

Tools: Google Colab with PyTorch

Allowed help: Your Section 2 reading notes and the PyTorch docs (pytorch.org)

Deliverable: One Colab notebook (.ipynb) with every cell run and all outputs visible

Deadline and submission: 03/11/26 noon 12pm via [google form link]

In [1]:
# Gear up:

#Run this cell first:

import math, random, re
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
seed = 42
torch.manual_seed(seed)
print('Seed:', seed)

Seed: 42


# Task 1: Telemetry check
Mary's sensors send a grid of readings: 4 time steps by 6 sensors.

In [2]:
telemetry = torch.randn(4, 6)

#### Using tensor operations only (no loops), print:

(a) the shape and dtype of telemetry

(b) the average reading of each sensor (6 numbers) with a single line using dim

(c) the time step (0 to 3) with the largest total reading, using sum and argmax

(d) telemetry reshaped to (2, 12) ; print its shape

(e) the readings after subtracting each sensor's average (broadcasting). Print the new column means rounded to 4 decimals; they should all be about 0

In [4]:
# (a) Print shape and dtype
print("(a) Shape:", telemetry.shape)
print("    Dtype:", telemetry.dtype)

(a) Shape: torch.Size([4, 6])
    Dtype: torch.float32


In [5]:
# (b) Average reading of each sensor
sensor_avg = telemetry.mean(dim=0)
print("\n(b) Average reading of each sensor:")
print(sensor_avg)


(b) Average reading of each sensor:
tensor([ 0.3941, -0.1619, -0.4131, -0.7680, -0.2383, -0.2645])


In [6]:
# (c) Time step with the largest total reading
max_time_step = telemetry.sum(dim=1).argmax()
print("\n(c) Time step with largest total reading:")
print(max_time_step.item())


(c) Time step with largest total reading:
0


In [7]:
# (d) Reshape telemetry to (2, 12)
reshaped = telemetry.reshape(2, 12)
print("\n(d) Reshaped telemetry shape:")
print(reshaped.shape)


(d) Reshaped telemetry shape:
torch.Size([2, 12])


In [8]:
# (e) Subtract each sensor's average using broadcasting
centered = telemetry - sensor_avg

print("\n(e) Column means after subtracting sensor averages:")
print(centered.mean(dim=0).round(decimals=4))


(e) Column means after subtracting sensor averages:
tensor([-0., 0., -0., -0., 0., 0.])


# Task 2: One neuron, one lesson

A single neuron looks at one sensor value x = 2.0 . The true answer is y = 1.0 (real danger).

In [9]:
x = torch.tensor(2.0)
y = torch.tensor(1.0)
w = torch.tensor(0.5, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

(a) Compute z = w*x + b , then a = sigmoid(z) , then loss = (a - y)**2 . Print z ,
a and loss .

(b) Call loss.backward() and print w.grad and b.grad .

(c) Check by hand (in plain Python, no autograd) with the chain-rule formula from the
notes: dL/dw = 2(a - y) * a(1 - a) * x . Print it next to w.grad . They must match.

(d) Run 3 steps of gradient descent (learning rate 0.5) in a loop. Remember
torch.no_grad() for the update and zeroing the gradients each step. Print the loss
after every step; it must decrease.

In [10]:
import math
import torch

# Initialize values
x = torch.tensor(2.0)
y = torch.tensor(1.0)

w = torch.tensor(0.5, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

# (a) Forward pass
z = w * x + b
a = torch.sigmoid(z)
loss = (a - y) ** 2

print("(a) Forward Pass:")
print("z:", z.item())
print("a:", a.item())
print("Loss:", loss.item())

(a) Forward Pass:
z: 1.0
a: 0.7310585975646973
Loss: 0.07232947647571564


In [11]:
# (b) Backward pass
loss.backward()

print("\n(b) Gradients using Autograd:")
print("w.grad:", w.grad.item())
print("b.grad:", b.grad.item())


(b) Gradients using Autograd:
w.grad: -0.2115083485841751
b.grad: -0.10575417429208755


In [12]:
# (c) Manual gradient calculation using chain rule
a_val = a.item()
y_val = y.item()
x_val = x.item()

manual_grad_w = 2 * (a_val - y_val) * a_val * (1 - a_val) * x_val

print("\n(c) Manual Gradient Check:")
print("Manual dL/dw:", manual_grad_w)
print("Autograd w.grad:", w.grad.item())
print("Gradients match:", math.isclose(manual_grad_w, w.grad.item(), rel_tol=1e-6))


(c) Manual Gradient Check:
Manual dL/dw: -0.2115083468329218
Autograd w.grad: -0.2115083485841751
Gradients match: True


In [13]:
# (d) Gradient descent for 3 steps
learning_rate = 0.5

print("\n(d) Gradient Descent:")

# Clear gradients from the earlier backward pass
w.grad.zero_()
b.grad.zero_()

for step in range(3):

    # Forward pass
    z = w * x + b
    a = torch.sigmoid(z)
    loss = (a - y) ** 2

    # Backward pass
    loss.backward()

    # Update parameters without tracking gradients
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad

    # Clear gradients
    w.grad.zero_()
    b.grad.zero_()

    print(f"Step {step+1}: Loss = {loss.item():.6f}")


(d) Gradient Descent:
Step 1: Loss = 0.072329
Step 2: Loss = 0.048497
Step 3: Loss = 0.035898


# Task 3: Decode the transmissions

Mission Control has intercepted 200 ship messages. Each one is labelled 1 = DISTRESS or 0 = ROUTINE. Run this cell as it is:

In [14]:
rng = random.Random(seed)

sections = [
    'engine bay', 'cargo hold', 'lab', 'bridge', 
    'airlock', 'reactor room', 'galley', 'sleep quarters'
]

distress_events = [
    'hull breach', 'coolant leak', 'fire', 'pressure loss', 
    'power failure', 'radiation spike', 'astrophage leak'
]

routine_events = [
    'system check', 'inventory count', 'water recycling',
    'sensor calibration', 'log backup', 'meal prep',
    'exercise cycle'
]

distress_templates = [
    'warning {ev} in {sec}', 
    'alarm {ev} detected near{sec}', 
    'urgent {ev} reported in {sec} crew respond', 
    '{ev} in {sec} evacuate now'
]

routine_templates = [
    '{ev} complete in {sec} all nominal', 
    'scheduled {ev} finished in {sec}', 
    'routine {ev} in {sec} no issues', 
    '{ev} started in {sec} as planned'
]

tricky_distress = [
    'calm report {ev} spreading in {sec} help needed'
]

tricky_routine = [
    'false alarm {ev} cleared in {sec}', 
    'drill only simulated {ev} in {sec} no danger'
]

p_distress = 0.40

def make_message():
    label = 1 if rng.random() < p_distress else 0
    tricky = rng.random() < 0.10

    if label == 1:
        ev = rng.choice(distress_events)
        tpl = rng.choice(
                tricky_distress if tricky else distress_templates
        )
    else:
        ev = rng.choice(
            distress_events if tricky else routine_events
        ) # tricky routine messages mention a distress event

        tpl = rng.choice(tricky_routine if tricky else routine_templates)

    return tpl.format(ev=ev, sec=rng.choice(sections)), label

messages = [make_message() for _ in range(200)]
train_data, val_data = messages[:160], messages[160:]
print(len(train_data), len(val_data))
print(sum(label for _, label in messages), 'distress messages out of', len(messages))
print(messages[:3])

160 40
78 distress messages out of 200
[('false alarm fire cleared in bridge', 0), ('warning radiation spike in galley', 1), ('calm report coolant leak spreading in bridge help needed', 1)]


Now teach Mary to read. The tokenizer is supplied; you write build_vocab and encode :

In [15]:
def tokenize(text):
    return re.findall(r"[a-z']+", text.lower())

PAD, UNK = '<pad>', '<unk>'
def build_vocab(data):
    # TODO: use ONLY the training messages. <pad> must get id 0 and <unk> id 1.

    # Initialize vocabulary with special tokens
    vocab = {PAD: 0, UNK: 1}

    # Build vocabulary using only training messages
    for text, label in data:
        tokens = tokenize(text)

        for token in tokens:
            if token not in vocab:
                vocab[token] = len(vocab)

    return vocab


def encode(text, vocab):
    #TODO: turn a message into a list of word ids (unknown words become <unk>)
    # Convert each token into its vocabulary ID
    return [vocab.get(token, vocab[UNK]) for token in tokenize(text)]

vocab = build_vocab(train_data)

print(len(vocab))
print(encode(train_data[0][0], vocab))
print(encode('mary hears a strange glorp', vocab))

79
[2, 3, 4, 5, 6, 7]
[1, 1, 1, 1, 1]


Then run Mission Control's supplied helpers ( Dataset , collate_fn and the data
loaders):

In [18]:
class TransmissionDataset(Dataset):
    def __init__(self, data, vocab):
        self.samples = [
            (encode(text, vocab), label) for text, label in data
            ]

    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        ids, label = self.samples[idx]
        return torch.tensor(ids, dtype=torch.long), torch.tensor(label,dtype=torch.long)
    
def collate_fn(batch):
    ids_list, labels = zip(*batch)
    padded = nn.utils.rnn.pad_sequence(list(ids_list), batch_first=True,padding_value=0)
    return padded, torch.stack(labels)

train_loader = DataLoader(TransmissionDataset(train_data, vocab), batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader= DataLoader(TransmissionDataset(val_data, vocab), batch_size=16, shuffle=False, collate_fn=collate_fn)

# Task 4: Build Mary's brain
Write an nn.Module named MaryNet with this design:
1. nn.Embedding(vocab_size, 16, padding_idx=0) turns word ids into vectors.
2. Mean pooling over the real words of each message, ignoring padding (use a mask).
3. nn.Linear(16, 16) , then ReLU, then nn.Dropout(0.3) .
4. nn.Linear(16, 2) giving raw logits (no softmax inside the model).
Then, in your notebook:
(a) Create the model, print(model) , and print the total number of parameters (use
numel() ).
(b) Verify that count by hand in code: V*16 + (16*16 + 16) + (16*2 + 2) where V
= len(vocab) .
(c) Push one batch from train_loader through the model and print the shapes of the
input batch and of the output. Expected shapes: (batch, longest_message) and
(batch, 2) .

In [19]:
import torch
import torch.nn as nn

# Define Mary's neural network
class MaryNet(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        # 1. Embedding layer
        self.embedding = nn.Embedding(vocab_size, 16, padding_idx=0)

        # 2. Fully connected layers
        self.fc1 = nn.Linear(16, 16)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

        # 3. Output layer
        self.fc2 = nn.Linear(16, 2)

    def forward(self, x):

        # Convert word IDs into embeddings
        embedded = self.embedding(x)

        # Create mask to ignore padding tokens
        mask = (x != 0).unsqueeze(-1).float()

        # Mean pooling over real words
        summed = (embedded * mask).sum(dim=1)
        lengths = mask.sum(dim=1).clamp(min=1)
        pooled = summed / lengths

        # Hidden layer
        out = self.fc1(pooled)
        out = self.relu(out)
        out = self.dropout(out)

        # Output raw logits
        logits = self.fc2(out)

        return logits


# (a) Create model and count parameters
model = MaryNet(len(vocab))

print("(a) Model Architecture:")
print(model)

total_params = sum(p.numel() for p in model.parameters())

print("\nTotal parameters:", total_params)

(a) Model Architecture:
MaryNet(
  (embedding): Embedding(79, 16, padding_idx=0)
  (fc1): Linear(in_features=16, out_features=16, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=16, out_features=2, bias=True)
)

Total parameters: 1570


In [20]:
# (b) Verify parameter count manually
V = len(vocab)

manual_params = V * 16 + (16 * 16 + 16) + (16 * 2 + 2)

print("\n(b) Manual Parameter Count:")
print("Vocabulary size:", V)
print("Manual parameters:", manual_params)
print("Counts match:", total_params == manual_params)


(b) Manual Parameter Count:
Vocabulary size: 79
Manual parameters: 1570
Counts match: True


In [21]:
# (c) Forward pass using one batch
batch_inputs, batch_labels = next(iter(train_loader))

outputs = model(batch_inputs)

print("\n(c) Forward Pass Shapes:")
print("Input batch shape:", batch_inputs.shape)
print("Output shape:", outputs.shape)


(c) Forward Pass Shapes:
Input batch shape: torch.Size([16, 7])
Output shape: torch.Size([16, 2])


# Task 5: Train Mary
Use nn.CrossEntropyLoss and torch.optim.Adam with lr=0.01 . Train for 25 epochs using the five-step training loop.

(a) Write an evaluate(model, loader) function that returns the average loss and the
accuracy (use model.eval() and torch.no_grad() ).

(b) Before training, print the validation loss. Print math.log(2) next to it. (A model
that is just guessing between 2 classes should start near that number.)

(c) Train, and print the training loss, validation loss and validation accuracy every 5
epochs.

In [22]:
import math
import torch
import torch.nn as nn

# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


# (a) Evaluation function
def evaluate(model, loader):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in loader:

            # Forward pass
            outputs = model(inputs)

            # Calculate loss
            loss = criterion(outputs, labels)

            # Accumulate loss
            total_loss += loss.item() * labels.size(0)

            # Predictions
            predictions = outputs.argmax(dim=1)

            # Count correct predictions
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy

In [23]:
# (b) Validation loss before training
initial_val_loss, initial_val_acc = evaluate(model, val_loader)

print("(b) Before Training:")
print(f"Initial Validation Loss: {initial_val_loss:.4f}")
print(f"math.log(2): {math.log(2):.4f}")
print(f"Initial Validation Accuracy: {initial_val_acc:.4f}")

(b) Before Training:
Initial Validation Loss: 0.7196
math.log(2): 0.6931
Initial Validation Accuracy: 0.2500


In [24]:
# (c) Training loop
epochs = 25

print("\n(c) Training Mary:")

for epoch in range(epochs):

    model.train()

    total_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels in train_loader:

        # Step 1: Zero gradients
        optimizer.zero_grad()

        # Step 2: Forward pass
        outputs = model(inputs)

        # Step 3: Compute loss
        loss = criterion(outputs, labels)

        # Step 4: Backward pass
        loss.backward()

        # Step 5: Update parameters
        optimizer.step()

        # Accumulate training loss
        total_train_loss += loss.item() * labels.size(0)
        total_train_samples += labels.size(0)

    # Average training loss
    avg_train_loss = total_train_loss / total_train_samples

    # Validation
    val_loss, val_acc = evaluate(model, val_loader)

    # Print results every 5 epochs
    if (epoch + 1) % 5 == 0:
        print(
            f"Epoch {epoch+1:2d}/{epochs} | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Accuracy: {val_acc*100:.2f}%"
        )


(c) Training Mary:
Epoch  5/25 | Train Loss: 0.0774 | Val Loss: 0.0155 | Val Accuracy: 100.00%
Epoch 10/25 | Train Loss: 0.0036 | Val Loss: 0.0004 | Val Accuracy: 100.00%
Epoch 15/25 | Train Loss: 0.0014 | Val Loss: 0.0001 | Val Accuracy: 100.00%
Epoch 20/25 | Train Loss: 0.0042 | Val Loss: 0.0001 | Val Accuracy: 100.00%
Epoch 25/25 | Train Loss: 0.0037 | Val Loss: 0.0000 | Val Accuracy: 100.00%


# Task 6: Mission report
Time for Mary's final exam.

(a) On the validation set, print the confusion matrix and the classification_report
(scikit-learn), using target_names=['routine', 'distress'] .

(b) Write a predict(text, model, vocab) function that returns the predicted label
and the two probabilities (softmax only to read the result). Run it on these three
transmissions and print the results:
1. urgent hull breach in cargo hold evacuate now
2. routine log backup in lab no issues
3. false alarm fire cleared in bridge

(c) Print one validation message Mary got wrong. (If she got none, print the message
she was least sure about.)

(d) Mary's reflection (3 to 4 sentences, in your own words): What is Mary's final
validation accuracy? Which kind of message confuses her most, and why might that
be?

In [26]:
# (a) Confusion Matrix and Classification Report

from sklearn.metrics import confusion_matrix, classification_report

model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for inputs, labels in val_loader:
        outputs = model(inputs)
        predictions = outputs.argmax(dim=1)

        y_true.extend(labels.tolist())
        y_pred.extend(predictions.tolist())

# Confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

print("(a) Confusion Matrix:")
print(cm)

# Classification report
print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    target_names=['routine', 'distress'],
    zero_division=0
))

(a) Confusion Matrix:
[[25  0]
 [ 0 15]]

Classification Report:
              precision    recall  f1-score   support

     routine       1.00      1.00      1.00        25
    distress       1.00      1.00      1.00        15

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40



In [29]:
# (b) Predict New Transmissions

import torch.nn.functional as F

def predict(text, model, vocab):

    model.eval()

    # Convert text into word IDs
    ids = encode(text, vocab)

    # Convert into tensor with batch dimension
    input_tensor = torch.tensor(
        [ids], dtype=torch.long
    )

    with torch.no_grad():

        # Forward pass
        logits = model(input_tensor)

        # Convert logits into probabilities
        probabilities = F.softmax(logits, dim=1)

        # Predicted label
        predicted_label = probabilities.argmax(dim=1).item()

    return predicted_label, probabilities[0].tolist()


# Three test transmissions
test_messages = [
    "urgent hull breach in cargo hold evacuate now",
    "routine log backup in lab no issues",
    "false alarm fire cleared in bridge"
]

print("(b) Predictions on New Transmissions:\n")

for text in test_messages:

    label, probs = predict(text, model, vocab)

    print("Message:", text)
    print("Predicted Label:", label)
    print("Predicted Class:", "DISTRESS" if label == 1 else "ROUTINE")
    print(f"Routine Probability: {probs[0]:.4f}")
    print(f"Distress Probability: {probs[1]:.4f}")
    print()

(b) Predictions on New Transmissions:

Message: urgent hull breach in cargo hold evacuate now
Predicted Label: 1
Predicted Class: DISTRESS
Routine Probability: 0.0000
Distress Probability: 1.0000

Message: routine log backup in lab no issues
Predicted Label: 0
Predicted Class: ROUTINE
Routine Probability: 1.0000
Distress Probability: 0.0000

Message: false alarm fire cleared in bridge
Predicted Label: 0
Predicted Class: ROUTINE
Routine Probability: 0.9998
Distress Probability: 0.0002



In [30]:
# (c) Find One Incorrectly Classified Validation Message

model.eval()

wrong_messages = []
least_confident_message = None
lowest_confidence = float('inf')

with torch.no_grad():

    for text, true_label in val_data:

        # Predict message
        pred_label, probs = predict(text, model, vocab)

        # Confidence of predicted class
        confidence = max(probs)

        # Store incorrect predictions
        if pred_label != true_label:
            wrong_messages.append(
                (text, true_label, pred_label, confidence)
            )

        # Track least confident prediction
        if confidence < lowest_confidence:
            lowest_confidence = confidence
            least_confident_message = (
                text, true_label, pred_label, confidence
            )


print("(c) Validation Message Analysis:\n")

if wrong_messages:

    text, true_label, pred_label, confidence = wrong_messages[0]

    print("Incorrectly Classified Message:")
    print("Message:", text)
    print("True Label:", true_label)
    print("Predicted Label:", pred_label)
    print(f"Confidence: {confidence:.4f}")

else:

    text, true_label, pred_label, confidence = least_confident_message

    print("No incorrectly classified messages found.")
    print("\nLeast Confident Prediction:")
    print("Message:", text)
    print("True Label:", true_label)
    print("Predicted Label:", pred_label)
    print(f"Confidence: {confidence:.4f}")

(c) Validation Message Analysis:

No incorrectly classified messages found.

Least Confident Prediction:
Message: false alarm radiation spike cleared in sleep quarters
True Label: 0
Predicted Label: 0
Confidence: 0.9996


In [31]:
# (d) Mary's Reflection

final_val_loss, final_val_acc = evaluate(model, val_loader)

print(f"Final Validation Loss: {final_val_loss:.4f}")
print(f"Final Validation Accuracy: {final_val_acc * 100:.2f}%")

print(f"Incorrectly Classified Messages: {len(wrong_messages)}")

Final Validation Loss: 0.0000
Final Validation Accuracy: 100.00%
Incorrectly Classified Messages: 0


In [33]:
print("=" * 45)
print("MISSION 2A: MARY'S FINAL REPORT")
print("=" * 45)

print(f"Training Messages   : {len(train_data)}")
print(f"Validation Messages : {len(val_data)}")
print(f"Vocabulary Size     : {len(vocab)}")
print(f"Model Parameters    : {sum(p.numel() for p in model.parameters())}")
print(f"Training Epochs     : 25")
print(f"Final Val Loss      : {final_val_loss:.4f}")
print(f"Final Val Accuracy  : {final_val_acc * 100:.2f}%")
print(f"Incorrect Messages  : {len(wrong_messages)}")

print("=" * 45)
print("Mary... think. Mary think!")
print("MISSION COMPLETE!")

MISSION 2A: MARY'S FINAL REPORT
Training Messages   : 160
Validation Messages : 40
Vocabulary Size     : 79
Model Parameters    : 1570
Training Epochs     : 25
Final Val Loss      : 0.0000
Final Val Accuracy  : 100.00%
Incorrect Messages  : 0
Mary... think. Mary think!
MISSION COMPLETE!


Mary achieved a final validation accuracy of **100%** on the validation dataset. She found **0** most confusing. This may be because some routine messages contain distress-related words such as "fire" or "alarm", making them similar to actual distress messages. Since the model uses mean pooling, it may struggle to understand the context and order of words in these messages.


#### Submission checklist

Notebook runs top to bottom (Runtime → Run all) with all outputs visible

Your own crew code is in the first cell

Every Log entry and reflection answer is written in a text cell, in your own words

Notebook shared with link access

"Mary... think. Mary think! Thank you, crew."
🚀